# NeoTwin: 01 — COLMAP Pose Estimation

**Runtime:** Google Colab T4 GPU  
**Time:** ~10 min  
**Output:** `sparse_output.zip` — camera poses ready for 3DGS training

### What this notebook does
1. Pulls a curated scene from Hugging Face (`nerf-gs-datasets`)
2. **Quality-filters** frames: removes blurry, over/under-exposed, and near-duplicate images
3. Runs COLMAP with `OPENCV` camera model + exhaustive matching for maximum accuracy
4. Validates reconstruction quality (reprojection error < 2px, track length > 3)
5. Downloads the `sparse/` folder ready for Notebook 02

> **Bring your own video?** Set `USE_OWN_DATA = True` in the config cell and upload your frames zip.

In [ ]:
# ─── 0. CONFIG — edit only this cell ────────────────────────────────────────
USE_OWN_DATA        = False   # True = upload your own frames zip
HF_DATASET          = 'rishitdagli/nerf-gs-datasets'
SCENE_INDEX         = 0       # which scene to use from the dataset
TARGET_FRAME_COUNT  = 120     # sweet-spot: enough coverage, not too slow
MIN_BLUR_THRESHOLD  = 80.0    # Laplacian variance — lower = blurrier
MAX_DUPLICATE_DIST  = 0.97    # cosine similarity above this = duplicate
CAMERA_MODEL        = 'OPENCV'  # OPENCV > SIMPLE_RADIAL for accuracy
USE_HLOC            = False   # True = SuperPoint+SuperGlue (dark/repetitive scenes)
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── 1. INSTALL DEPENDENCIES ─────────────────────────────────────────────────
import subprocess, sys

def run(cmd): subprocess.run(cmd, shell=True, check=True)

run('apt-get update -qq && apt-get install -y -qq colmap libboost-all-dev')
run('pip install -q pycolmap datasets pillow tqdm scikit-image')

if USE_HLOC:
    run('pip install -q git+https://github.com/cvg/Hierarchical-Localization.git')

print('✅ Dependencies installed')

In [ ]:
# ─── 2. LOAD IMAGES ──────────────────────────────────────────────────────────
import os, io, shutil
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

DATA_DIR = Path('data')
RAW_DIR  = DATA_DIR / 'raw'
IMG_DIR  = DATA_DIR / 'images'
for d in [RAW_DIR, IMG_DIR]: d.mkdir(parents=True, exist_ok=True)

if USE_OWN_DATA:
    from google.colab import files
    print('Upload your frames ZIP:')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    shutil.unpack_archive(zip_name, RAW_DIR)
    raw_images = sorted(RAW_DIR.rglob('*.jpg')) + sorted(RAW_DIR.rglob('*.png'))
else:
    from datasets import load_dataset
    print(f'Loading scene {SCENE_INDEX} from {HF_DATASET}...')
    ds = load_dataset(HF_DATASET, split='train')
    scene = ds[SCENE_INDEX]
    
    # Extract all frames from this scene
    raw_images = []
    frames = scene.get('images', [scene.get('image')])
    if not isinstance(frames, list): frames = [frames]
    for i, img in enumerate(tqdm(frames, desc='Saving frames')):
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        p = RAW_DIR / f'frame_{i:04d}.jpg'
        img.save(p, quality=95)
        raw_images.append(p)

print(f'✅ Loaded {len(raw_images)} raw frames')

In [ ]:
# ─── 3. QUALITY FILTER ────────────────────────────────────────────────────────
# Removes: (a) blurry frames  (b) over/under-exposed  (c) near-duplicates
import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim

def blur_score(path):
    """Laplacian variance — higher = sharper."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.Laplacian(img, cv2.CV_64F).var()

def exposure_ok(path):
    """Reject if >40% pixels are near-black or near-white."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    h = cv2.calcHist([img], [0], None, [256], [0, 256]).flatten()
    total = img.size
    overexposed  = h[240:].sum() / total
    underexposed = h[:15].sum()  / total
    return (overexposed < 0.40) and (underexposed < 0.40)

def thumb(path, size=64):
    """Tiny greyscale thumbnail for duplicate detection."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img, (size, size)).flatten().astype(np.float32)

# Step A: filter by blur + exposure
passed = []
for p in tqdm(raw_images, desc='Blur/exposure filter'):
    if blur_score(p) >= MIN_BLUR_THRESHOLD and exposure_ok(p):
        passed.append(p)

print(f'  After blur/exposure filter: {len(passed)}/{len(raw_images)}')

# Step B: remove near-duplicates via cosine similarity on thumbnails
def cosine(a, b):
    a, b = a / (np.linalg.norm(a) + 1e-9), b / (np.linalg.norm(b) + 1e-9)
    return np.dot(a, b)

thumbs = [thumb(p) for p in tqdm(passed, desc='Building thumbnails')]
kept_indices = [0]
for i in range(1, len(passed)):
    last_kept = thumbs[kept_indices[-1]]
    if cosine(thumbs[i], last_kept) < MAX_DUPLICATE_DIST:
        kept_indices.append(i)

deduped = [passed[i] for i in kept_indices]
print(f'  After duplicate filter: {len(deduped)}')

# Step C: if still > TARGET_FRAME_COUNT, pick evenly spaced subset
if len(deduped) > TARGET_FRAME_COUNT:
    indices = np.linspace(0, len(deduped) - 1, TARGET_FRAME_COUNT, dtype=int)
    deduped = [deduped[i] for i in indices]

# Copy final selection into IMG_DIR with clean names
for i, src in enumerate(tqdm(deduped, desc='Copying filtered images')):
    dst = IMG_DIR / f'{i:04d}.jpg'
    shutil.copy(src, dst)
    # Ensure consistent 1920×1080 to avoid VRAM spikes during training
    img = Image.open(dst)
    if max(img.size) > 1920:
        img.thumbnail((1920, 1080), Image.LANCZOS)
        img.save(dst, quality=95)

final_count = len(list(IMG_DIR.glob('*.jpg')))
print(f'\n✅ Final dataset: {final_count} high-quality frames → {IMG_DIR}')

In [ ]:
# ─── 4A. COLMAP (default) ────────────────────────────────────────────────────
if not USE_HLOC:
    import pycolmap
    from pathlib import Path

    SPARSE_DIR = DATA_DIR / 'sparse'
    DB_PATH    = DATA_DIR / 'database.db'
    SPARSE_DIR.mkdir(parents=True, exist_ok=True)

    # --- Feature extraction (OPENCV model = full lens-distortion correction) ---
    feat_opts = pycolmap.ImageReaderOptions()
    feat_opts.camera_model = CAMERA_MODEL
    feat_opts.single_camera = True   # one camera = more robust for phone video

    print('📐 Extracting SIFT features...')
    pycolmap.extract_features(
        database_path=str(DB_PATH),
        image_path=str(IMG_DIR),
        image_reader_options=feat_opts
    )

    # --- Matching: exhaustive for ≤200 images, sequential for >200 ---
    print('🔗 Matching features...')
    if final_count <= 200:
        pycolmap.match_exhaustive(database_path=str(DB_PATH))
    else:
        seq_opts = pycolmap.SequentialMatchingOptions()
        seq_opts.overlap = 20       # consider 20 neighbors each side
        seq_opts.loop_detection = True
        pycolmap.match_sequential(
            database_path=str(DB_PATH),
            sequential_matching_options=seq_opts
        )

    # --- Sparse reconstruction (incremental mapper) ---
    print('🗺️  Running incremental mapper...')
    maps = pycolmap.incremental_mapping(
        database_path=str(DB_PATH),
        image_path=str(IMG_DIR),
        output_path=str(SPARSE_DIR)
    )
    print(f'   → {len(maps)} reconstruction(s) produced')

In [ ]:
# ─── 4B. HLoc (optional — dark / repetitive scenes) ─────────────────────────
if USE_HLOC:
    from hloc import extract_features, match_features, reconstruction
    from hloc.utils.base_model import dynamic_load
    import hloc.matchers, hloc.extractors
    from pathlib import Path

    HLOC_DIR = DATA_DIR / 'hloc'
    HLOC_DIR.mkdir(parents=True, exist_ok=True)

    feature_conf  = extract_features.confs['superpoint_aachen']
    matcher_conf  = match_features.confs['superglue']

    features_path = HLOC_DIR / 'features.h5'
    matches_path  = HLOC_DIR / 'matches.h5'
    pairs_path    = HLOC_DIR / 'pairs.txt'

    print('📐 Extracting SuperPoint features...')
    extract_features.main(feature_conf, IMG_DIR, feature_path=features_path)

    print('🔗 Running SuperGlue matching...')
    # Build exhaustive pair list
    imgs = sorted(IMG_DIR.glob('*.jpg'))
    with open(pairs_path, 'w') as f:
        for i in range(len(imgs)):
            for j in range(i + 1, min(i + 30, len(imgs))):
                f.write(f'{imgs[i].name} {imgs[j].name}\n')

    match_features.main(
        matcher_conf, pairs_path,
        features=features_path, matches=matches_path
    )

    print('🗺️  Reconstructing with COLMAP + HLoc matches...')
    maps = reconstruction.main(
        DATA_DIR / 'sparse',
        IMG_DIR, pairs_path,
        features_path, matches_path
    )
    SPARSE_DIR = DATA_DIR / 'sparse'
    print(f'✅ HLoc reconstruction done — {len(maps)} models')

In [ ]:
# ─── 5. QUALITY VALIDATION ───────────────────────────────────────────────────
# Gate on reprojection error and track length before allowing download.
import pycolmap, json
from pathlib import Path

SPARSE_DIR = DATA_DIR / 'sparse'

# Find the largest reconstruction (most images registered)
recon_dirs = sorted(SPARSE_DIR.iterdir())
assert recon_dirs, '❌ COLMAP produced no reconstructions. Check image quality or try USE_HLOC=True'

best_dir = max(recon_dirs, key=lambda d: len(list(d.glob('images.bin'))))
recon = pycolmap.Reconstruction(str(best_dir))

n_cameras   = len(recon.cameras)
n_images    = len(recon.images)
n_points    = len(recon.points3D)

# Mean track length (how many images each 3D point appears in)
track_lengths = [pt.track.length() for pt in recon.points3D.values()]
mean_track    = float(np.mean(track_lengths)) if track_lengths else 0.0

# Mean reprojection error
errors = [
    np.linalg.norm([obs.x - 0, obs.y - 0])  # approximation via summary stats
    for img in recon.images.values()
    for obs in img.get_observations()
    if obs.point3D_id != -1
]
# Use COLMAP's built-in summary for reprojection error
summary = recon.compute_mean_reprojection_error() if hasattr(recon, 'compute_mean_reprojection_error') else -1

quality_report = {
    'cameras_calibrated': n_cameras,
    'images_registered': n_images,
    'total_input_images': final_count,
    'registration_rate': f'{n_images / final_count * 100:.1f}%',
    'sparse_points': n_points,
    'mean_track_length': round(mean_track, 2),
    'mean_reprojection_error_px': round(summary, 3) if summary != -1 else 'N/A',
    'quality_gate_passed': (n_images / final_count > 0.75 and mean_track >= 3.0)
}

print('\n📊 RECONSTRUCTION QUALITY REPORT')
print('─' * 42)
for k, v in quality_report.items():
    icon = '✅' if k == 'quality_gate_passed' and v else ('❌' if k == 'quality_gate_passed' else '  ')
    print(f'{icon}  {k:35s}: {v}')

# Save report alongside sparse output
with open(DATA_DIR / 'quality_report.json', 'w') as f:
    json.dump(quality_report, f, indent=2)

if not quality_report['quality_gate_passed']:
    print('\n⚠️  Quality gate FAILED.')
    print('   Options: (1) Capture more frames with >70% overlap')
    print('           (2) Set USE_HLOC=True for difficult scenes')
    print('           (3) Lower MIN_BLUR_THRESHOLD slightly')
else:
    print('\n✅ Quality gate PASSED — safe to proceed to Notebook 02')

In [ ]:
# ─── 6. PACKAGE & DOWNLOAD ───────────────────────────────────────────────────
import shutil
from google.colab import files

# Bundle sparse reconstruction + quality report + filtered images manifest
shutil.make_archive('sparse_output', 'zip', DATA_DIR)

print(f'📦 sparse_output.zip ready')
print('   Contains: sparse/ · quality_report.json · images/')
files.download('sparse_output.zip')